In [3]:
import os
# Set a path for your local directory
local_folder_path = "./data"

# Ensure the directory exists, or create it
if not os.path.exists(local_folder_path):
    os.makedirs(local_folder_path)


# Coalesce the data to reduce the number of output files (e.g., coalesce to 1 file for small datasets)

num_partitions = 1  

users_df = users_df.coalesce(num_partitions)
preferences_df = preferences_df.coalesce(num_partitions)
devices_df = devices_df.coalesce(num_partitions)
membership_df = membership_df.coalesce(num_partitions)


# Write DataFrames to Parquet in the local folder using Snappy compression 
users_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/users/", compression="snappy")

preferences_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/preferences/", compression="snappy")

devices_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/devices/", compression="snappy")

membership_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/membership/", compression="snappy")


print("Data successfully written to local folder in Parquet format with Snappy compression.")

Data successfully written to local folder in Parquet format with Snappy compression.


In [8]:
!pip install faker


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.5 MB/s eta 0:00:00a 0:00:01


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col
import random
import json
from faker import Faker

# Initialize Spark session
spark = SparkSession.builder \
    .appName("NikeUsersDataGeneration") \
    .getOrCreate()

# Initialize Faker instance
fake = Faker()

# Function to generate sample users
def generate_sample_users(num_records):
    users = []
    domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com"]  # Realistic domains
    
    for i in range(num_records):
        first_name = fake.first_name()
        last_name = fake.last_name()
        username = f"{first_name.lower()}.{last_name.lower()}"  # Constructing realistic usernames
        email = f"{username}@{random.choice(domains)}"  # Combining username with random domain
        dob = fake.date_of_birth(minimum_age=15, maximum_age=90).strftime("%Y-%m-%d")  # Random DOB
        shopping_preference = fake.random_element(elements=('In-Store', 'Online', 'Both'))
        location = fake.city()  # Location
        phone_number = fake.phone_number()

        users.append((i + 1, first_name, last_name, email, dob, shopping_preference, location, phone_number))
    
    return users

# Create the users DataFrame
num_users = 1000000  
user_data = generate_sample_users(num_users)
user_columns = ['user_id', 'first_name', 'last_name', 'email', 'dob', 'shopping_preference', 'location', 'phone_number']
users_df = spark.createDataFrame(user_data, user_columns)

# Show the generated users data
users_df.show()

# Function to generate sample preferences
def generate_sample_preferences(user_ids, num_records):
    preferences = []
    for user_id in user_ids:
        sport_interest = json.dumps(fake.words(nb=random.randint(1, 3), unique=True, ext_word_list=['Soccer', 'Basketball', 'Tennis', 'Running', 'Baseball','Swimming','Training&Gym','Dance','Football','Golf','Softball','Yoga','Volleyball','Skateboarding']))
        brand_interest = json.dumps(fake.words(nb=random.randint(1, 2), unique=True, ext_word_list=['Nike', 'Jordan', 'Converse', 'Kobe','Air Force 1','Air Max','ACG','Big&Tall','Blazer','Maternity','N7','Nike By You','Nike Sportswear','NikeLab','SNKRS']))
        shoe_size = fake.random_element(elements=['4','4.5','5','5.5','6','6.5','7','7.5', '8','8.5', '9', '10', '11','11.5','12','13','14','15','16'])
        favorite_teams = json.dumps(fake.words(nb=random.randint(1, 2), unique=True, ext_word_list=['Atlanta Hawks','Boston Celtics','Phoenix Suns','Brooklyn Nets','Charlotte Hornets','Chicago Bulls','Cleveland Cavaliers','Denver Nuggets','Golden State Warriors','Houstan Rockets','Indiana Pacers','Los Angeles Clippers','Miami Heat','New York Knicks','Lakers', 'Bulls', 'Warriors', 'Heat']))
        favorite_cities = json.dumps(fake.words(nb=random.randint(1, 2), unique=True, ext_word_list=['New York City', 'Los Angeles', 'Chicago']))
        favorite_athletes = json.dumps(fake.words(nb=random.randint(1, 2), unique=True, ext_word_list=['LeBron James', 'Serena Williams', 'Michael Jordan', 'Tom Brady', 'Naomi Osaka','Erling Haaland','Cristiano Ronaldo','Marcus Rashford','Devin Booker','Ja Morant','Jayson Tatum','Kevin Durant','Russell Westbrook']))
        
        preferences.append((user_id, sport_interest, brand_interest, shoe_size, favorite_teams, favorite_cities, favorite_athletes))
    
    return preferences

# Generate sample preferences for the users
preferences_data = generate_sample_preferences([user_id for user_id in range(1, num_users + 1)], num_users)
preferences_columns = ['user_id', 'sport_interest', 'brand_interest', 'shoe_size', 'favorite_teams', 'favorite_cities', 'favorite_athletes']
preferences_df = spark.createDataFrame(preferences_data, preferences_columns)

# Show the generated preferences data
preferences_df.show()

# Function to generate sample devices
def generate_sample_devices(user_ids, num_records):
    device_types = ['Phone', 'Tablet', 'Laptop', 'Desktop']
    device_os = ['iOS', 'Android', 'Windows', 'macOS']
    
    devices = []
    for user_id in user_ids:
        device_type = fake.random_element(elements=device_types)
        device_os_type = fake.random_element(elements=device_os)
        device_name = f"{device_type} {random.randint(1, 10)}"
        
        devices.append((user_id, device_type, device_os_type, device_name))
    
    return devices

# Generate sample devices for the users
devices_data = generate_sample_devices([user_id for user_id in range(1, num_users + 1)], num_users)
devices_columns = ['user_id', 'device_type', 'device_os', 'device_name']
devices_df = spark.createDataFrame(devices_data, devices_columns)

# Show the generated devices data
devices_df.show()

# Function to generate sample membership data
def generate_sample_membership(user_ids, num_records):
    membership_types = ['Silver', 'Gold', 'Platinum']
    
    membership = []
    for user_id in user_ids:
        membership_type = fake.random_element(elements=membership_types)
        points = random.randint(0, 5000)
        
        membership.append((user_id, membership_type, points))
    
    return membership

# Generate sample membership for the users
membership_data = generate_sample_membership([user_id for user_id in range(1, num_users + 1)], num_users)
membership_columns = ['user_id', 'membership_type', 'points']
membership_df = spark.createDataFrame(membership_data, membership_columns)

# Show the generated membership data
membership_df.show()


